# Typhoon OCR 1.5 to CSV on Google Colab

โน้ตบุ๊กนี้ใช้ `Typhoon OCR 1.5` เพื่ออ่านภาพเอกสารจาก Google Drive แล้วแปลงผลลัพธ์เป็นไฟล์ CSV ที่มีคอลัมน์ `id`, `id_doc`, `row_num`, `party_name`, `vote` โดย `vote` จะถูกแปลงให้เหลือเฉพาะเลขอารบิกเท่านั้น

In [ ]:
!pip -q install typhoon-ocr

In [ ]:
from google.colab import drive
from getpass import getpass
import os

drive.mount('/content/drive')

if not os.environ.get('TYPHOON_OCR_API_KEY'):
    os.environ['TYPHOON_OCR_API_KEY'] = getpass('Typhoon OCR API key: ')

print('Drive mounted and API key is ready.')

In [ ]:
from pathlib import Path

# ปรับ path นี้ให้ตรงกับตำแหน่งโฟลเดอร์ใน Google Drive ของคุณ
INPUT_DIR = Path('/content/drive/MyDrive/super-ai-engineer-season-6-ocr-2569/data/images')
OUTPUT_DIR = Path('/content/drive/MyDrive/typhoon_ocr_output')
OUTPUT_CSV = OUTPUT_DIR / 'ocr_results.csv'
CACHE_DIR = OUTPUT_DIR / 'ocr_cache'

# ตั้งค่าเพื่อควบคุมการประมวลผล
SLEEP_SECONDS = 3.2
RETRIES = 3
OVERWRITE_CACHE = False
MAX_FILES = None  # เช่น 5 ถ้าต้องการทดลองกับบางไฟล์ก่อน

print('INPUT_DIR =', INPUT_DIR)
print('OUTPUT_CSV =', OUTPUT_CSV)
print('CACHE_DIR =', CACHE_DIR)
print('INPUT_DIR exists =', INPUT_DIR.exists())

## Helper functions

cell ถัดไปประกอบด้วย logic ทั้งหมดของงาน:

- แปลงเลขไทยเป็นเลขอารบิก
- สร้าง `id_doc` จากชื่อไฟล์
- เรียก `Typhoon OCR 1.5`
- parse ตารางเป็นแถวข้อมูล
- เขียนผลลัพธ์ลง CSV

In [ ]:
from __future__ import annotations

import csv
import re
import time
from dataclasses import dataclass
from typing import Iterable

THAI_DIGITS = str.maketrans('๐๑๒๓๔๕๖๗๘๙', '0123456789')
SUPPORTED_SUFFIXES = {'.png', '.jpg', '.jpeg', '.pdf'}
PAGE_SUFFIX_PATTERN = re.compile(r'^(?P<base>.+?)(?:_page(?P<page>\d+))?$', re.IGNORECASE)
TABLE_SEPARATOR_PATTERN = re.compile(r'^[\s\-\|\:]+$')


@dataclass(frozen=True)
class ParsedRow:
    id_doc: str
    row_num: str
    party_name: str
    vote: str
    source_file: str


def normalize_text(text: str) -> str:
    text = text.replace('\u200b', ' ')
    text = text.replace('<br>', ' ')
    return re.sub(r'\s+', ' ', text).strip()


def thai_to_arabic(text: str) -> str:
    return text.translate(THAI_DIGITS)


def digits_only(text: str) -> str:
    normalized = thai_to_arabic(text)
    return ''.join(re.findall(r'\d+', normalized))


def natural_sort_key(text: str) -> list[object]:
    return [int(part) if part.isdigit() else part.lower() for part in re.findall(r'\d+|\D+', text)]


def derive_id_doc(file_path: Path) -> str:
    match = PAGE_SUFFIX_PATTERN.match(file_path.stem)
    if not match:
        return file_path.stem
    return match.group('base')


def derive_page_num(file_path: Path) -> int:
    match = PAGE_SUFFIX_PATTERN.match(file_path.stem)
    if not match or not match.group('page'):
        return 1
    return int(match.group('page'))


def iter_input_files(input_dir: Path) -> list[Path]:
    if not input_dir.exists():
        raise FileNotFoundError(f'Input folder does not exist: {input_dir}')
    if not input_dir.is_dir():
        raise NotADirectoryError(f'Input path is not a folder: {input_dir}')

    files = [path for path in input_dir.iterdir() if path.is_file() and path.suffix.lower() in SUPPORTED_SUFFIXES]
    return sorted(
        files,
        key=lambda path: (
            natural_sort_key(derive_id_doc(path)),
            derive_page_num(path),
            path.name.lower(),
        ),
    )


def get_ocr_markdown(pdf_or_image_path: Path, cache_path: Path, overwrite_cache: bool, retries: int) -> str:
    if cache_path.exists() and not overwrite_cache:
        return cache_path.read_text(encoding='utf-8')

    from typhoon_ocr import ocr_document

    last_error = None
    for attempt in range(1, retries + 1):
        try:
            markdown = str(ocr_document(pdf_or_image_path=str(pdf_or_image_path)))
            cache_path.parent.mkdir(parents=True, exist_ok=True)
            cache_path.write_text(markdown, encoding='utf-8')
            return markdown
        except Exception as exc:
            last_error = exc
            wait_seconds = min(12.0, 2.0 * attempt)
            print(f'[WARN] OCR failed for {pdf_or_image_path.name} (attempt {attempt}/{retries}): {exc}')
            if attempt < retries:
                time.sleep(wait_seconds)

    raise RuntimeError(f'OCR failed for {pdf_or_image_path.name}: {last_error}')


def is_table_separator(line: str) -> bool:
    return bool(TABLE_SEPARATOR_PATTERN.fullmatch(line.strip()))


def looks_like_header(cells: list[str]) -> bool:
    joined = ' '.join(cell.lower() for cell in cells)
    header_keywords = (
        'หมายเลข',
        'ชื่อชื่อสกุล',
        'ชื่อ - ชื่อสกุล',
        'ผู้สมัคร',
        'ผู้สมัครรับเลือกตั้ง',
        'สังกัด',
        'พรรคการเมือง',
        'ได้คะแนน',
    )
    return any(keyword in joined for keyword in header_keywords)


def parse_markdown_table(markdown: str, id_doc: str, source_file: str) -> list[ParsedRow]:
    rows: list[ParsedRow] = []
    seen: set[tuple[str, str, str]] = set()

    for raw_line in markdown.splitlines():
        line = raw_line.strip()
        if '|' not in line:
            continue
        if is_table_separator(line):
            continue

        cells = [normalize_text(cell) for cell in line.strip('|').split('|')]
        if len(cells) < 4:
            continue
        if looks_like_header(cells):
            continue

        row_num = digits_only(cells[0])
        party_name = normalize_text(cells[-2])
        vote = digits_only(cells[-1])
        if not row_num or not party_name or not vote:
            continue

        record_key = (row_num, party_name, vote)
        if record_key in seen:
            continue
        seen.add(record_key)

        rows.append(
            ParsedRow(
                id_doc=id_doc,
                row_num=row_num,
                party_name=party_name,
                vote=vote,
                source_file=source_file,
            )
        )

    return rows


def parse_plain_text(markdown: str, id_doc: str, source_file: str) -> list[ParsedRow]:
    rows: list[ParsedRow] = []
    seen: set[tuple[str, str, str]] = set()

    for raw_line in markdown.splitlines():
        line = normalize_text(raw_line)
        if not line:
            continue
        if '|' in line:
            continue

        parts = [normalize_text(part) for part in re.split(r'\s{2,}', line) if normalize_text(part)]
        if len(parts) < 3:
            continue

        row_num = digits_only(parts[0])
        party_name = normalize_text(parts[-2])
        vote = digits_only(parts[-1])
        if not row_num or not party_name or not vote:
            continue

        record_key = (row_num, party_name, vote)
        if record_key in seen:
            continue
        seen.add(record_key)

        rows.append(
            ParsedRow(
                id_doc=id_doc,
                row_num=row_num,
                party_name=party_name,
                vote=vote,
                source_file=source_file,
            )
        )

    return rows


def extract_rows(markdown: str, id_doc: str, source_file: str) -> list[ParsedRow]:
    rows = parse_markdown_table(markdown, id_doc=id_doc, source_file=source_file)
    if rows:
        return rows
    return parse_plain_text(markdown, id_doc=id_doc, source_file=source_file)


def write_csv(rows: Iterable[ParsedRow], output_csv: Path) -> None:
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    with output_csv.open('w', encoding='utf-8-sig', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=['id', 'id_doc', 'row_num', 'party_name', 'vote'])
        writer.writeheader()
        for index, row in enumerate(rows, start=1):
            writer.writerow(
                {
                    'id': index,
                    'id_doc': row.id_doc,
                    'row_num': row.row_num,
                    'party_name': row.party_name,
                    'vote': row.vote,
                }
            )


def run_pipeline(
    input_dir: Path,
    output_csv: Path,
    cache_dir: Path,
    sleep_seconds: float,
    retries: int,
    overwrite_cache: bool,
    max_files: int | None = None,
) -> list[ParsedRow]:
    input_files = iter_input_files(input_dir)
    if max_files is not None:
        input_files = input_files[:max_files]

    print(f'[INFO] Found {len(input_files)} files in {input_dir}')
    all_rows: list[ParsedRow] = []

    for index, input_file in enumerate(input_files, start=1):
        id_doc = derive_id_doc(input_file)
        cache_path = cache_dir / f'{input_file.stem}.md'
        cache_hit = cache_path.exists() and not overwrite_cache

        print(f'[INFO] ({index}/{len(input_files)}) OCR: {input_file.name}')
        markdown = get_ocr_markdown(
            pdf_or_image_path=input_file,
            cache_path=cache_path,
            overwrite_cache=overwrite_cache,
            retries=retries,
        )
        extracted_rows = extract_rows(markdown=markdown, id_doc=id_doc, source_file=input_file.name)
        print(f'[INFO]     Extracted {len(extracted_rows)} rows from {input_file.name}')
        all_rows.extend(extracted_rows)

        if not cache_hit and index < len(input_files):
            time.sleep(sleep_seconds)

    write_csv(all_rows, output_csv)
    print(f'[INFO] Wrote {len(all_rows)} rows to {output_csv}')
    return all_rows

In [ ]:
rows = run_pipeline(
    input_dir=INPUT_DIR,
    output_csv=OUTPUT_CSV,
    cache_dir=CACHE_DIR,
    sleep_seconds=SLEEP_SECONDS,
    retries=RETRIES,
    overwrite_cache=OVERWRITE_CACHE,
    max_files=MAX_FILES,
)

print('Total parsed rows =', len(rows))

In [ ]:
import pandas as pd

df = pd.read_csv(OUTPUT_CSV)
display(df.head(20))
print('CSV path =', OUTPUT_CSV)
print('Total rows in CSV =', len(df))

In [ ]:
# Optional: ดาวน์โหลดไฟล์ CSV กลับมาที่เครื่อง
# from google.colab import files
# files.download(str(OUTPUT_CSV))

In [4]:
import os

# PyPDF2 ไม่ต้องใช้ poppler-utils
!pip install pypdf openai pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 10.1 MB/s eta 0:00:00


In [9]:
import os
import pandas as pd
import pypdf # ใช้ pypdf แทน PyPDF2
from openai import OpenAI
import re
from getpass import getpass

In [10]:
if not os.environ.get('TYPHOON_OCR_API_KEY'):
    os.environ['TYPHOON_OCR_API_KEY'] = getpass('Typhoon OCR API key: ')

In [11]:
# Initialize OpenAI client after API key is set
client = OpenAI(api_key=os.environ['TYPHOON_OCR_API_KEY'], base_url="https://api.typhoon-ocr.ai/v1")

MODEL_NAME = "typhoon-v1.5x-vision-instruct"

pdf_path = 'ocr-voter (1).pdf'
template_path = 'summission_template2.csv'

# ฟังก์ชันแปลงตัวเลขไทยเป็นอารบิก (เผื่อโมเดลหลุดตัวเลขไทยมา)
def thai_to_arabic(text):
    thai_digits = '๐๑๒๓๔๕๖๗๘๙'
    arabic_digits = '0123456789'
    translation_table = str.maketrans(thai_digits, arabic_digits)
    return text.translate(translation_table)

# --- 2. อ่านไฟล์ Template ---
df_template = pd.read_csv(template_path)
# สร้าง dictionary เพื่อเก็บผลลัพธ์ โดยใช้ id เป็น key
results_dict = {}

# --- 3. แปลง PDF และเรียกใช้ OCR ---
extracted_texts = []
try:
    with open(pdf_path, 'rb') as f:
        reader = pypdf.PdfReader(f)
        num_pages = len(reader.pages)
        print(f"กำลังประมวลผล PDF จำนวน {num_pages} หน้า...")
        for i, page in enumerate(reader.pages):
            text = page.extract_text()
            if text:
                extracted_texts.append(text)
            else:
                extracted_texts.append("") # Append empty string if no text
except Exception as e:
    print(f"Error reading PDF with pypdf: {e}")
    # หากเกิดข้อผิดพลาดในการอ่าน PDF ด้วย pypdf อาจจะต้องตรวจสอบไฟล์ หรือลองวิธีอื่น

if not extracted_texts:
    print("ไม่สามารถดึงข้อความจาก PDF ได้ อาจเป็นเพราะไฟล์ว่างเปล่า เสียหาย หรือไม่มีข้อความที่สามารถอ่านได้")

for i, page_text in enumerate(extracted_texts):
    if not page_text:
        print(f"Skipping empty Page {i+1}...")
        continue

    print(f"Processing Page {i+1}...")

    # ส่ง Prompt ให้ Typhoon 1.5 - ตอนนี้ใช้ข้อความแทนรูปภาพ
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": f"ช่วยดึงข้อมูลคะแนนจากข้อความนี้ โดยให้ผลลัพธ์เป็นตารางที่มีคอลัมน์ row_num และ votes (คะแนน) โดย votes ต้องเป็นตัวเลขอารบิกเท่านั้น หากไม่มีคะแนนให้ใส่ 0 ไม่ต้องเขียนคำบรรยายใดๆ เพิ่มเติม:\n\n{page_text}"}
                ],
            }
        ],
        max_tokens=1000,
    )

    output_text = response.choices[0].message.content
    print(f"Output from AI:\n{output_text}")

    # คลีนข้อมูลเบื้องต้นและบันทึกลงในผลลัพธ์
    # (ในขั้นตอนนี้อาจต้องใช้ Regex เพื่อดึงตัวเลขออกมาให้แม่นยำขึ้น)
    # ตัวอย่างการดึงข้อมูลแบบง่าย (ขึ้นอยู่กับ Format ที่โมเดลตอบกลับมา)
    lines = output_text.split('\n')
    for line in lines:
        match = re.search(r'(\d+)\s+(\d+)', thai_to_arabic(line))
        if match:
            row_num, votes = match.groups()
            # เก็บข้อมูลไว้จับคู่กับ id ในขั้นตอนสุดท้าย
            # หมายเหตุ: การจับคู่ doc_id และ row_num ขึ้นอยู่กับลำดับหน้าใน PDF
            results_dict[int(row_num)] = int(votes)

# --- 4. รวมข้อมูลเข้ากับ Template และบันทึกผล ---
# สมมติว่า row_num ในหน้าแรกๆ ตรงกับลำดับใน csv
df_template['votes'] = df_template['row_num'].map(results_dict).fillna(0).astype(int)

# บันทึกไฟล์
df_template.to_csv('submission_output.csv', index=False)
print("บันทึกไฟล์เรียบร้อยแล้ว: submission_output.csv")

กำลังประมวลผล PDF จำนวน 846 หน้า...
Skipping empty Page 1...
Skipping empty Page 2...
Skipping empty Page 3...
Skipping empty Page 4...
Skipping empty Page 5...
Skipping empty Page 6...
Skipping empty Page 7...
Skipping empty Page 8...
Skipping empty Page 9...
Skipping empty Page 10...
Skipping empty Page 11...
Skipping empty Page 12...
Skipping empty Page 13...
Skipping empty Page 14...
Skipping empty Page 15...
Skipping empty Page 16...
Skipping empty Page 17...
Skipping empty Page 18...
Skipping empty Page 19...
Skipping empty Page 20...
Skipping empty Page 21...
Skipping empty Page 22...
Skipping empty Page 23...
Skipping empty Page 24...
Skipping empty Page 25...
Skipping empty Page 26...
Skipping empty Page 27...
Skipping empty Page 28...
Skipping empty Page 29...
Skipping empty Page 30...
Skipping empty Page 31...
Skipping empty Page 32...
Skipping empty Page 33...
Skipping empty Page 34...
Skipping empty Page 35...
Skipping empty Page 36...
Skipping empty Page 37...
Skipping em

APIConnectionError: Connection error.